In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFTGate

# 1. Función del Oráculo
def c_amod15(a, power):
    if a not in [2, 7, 8, 11, 13]:
        raise ValueError("'a' debe ser 2, 7, 8, 11 o 13")
    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2,13]: U.swap(0,1); U.swap(1,2); U.swap(2,3)
        if a in [7,8]: U.swap(2,3); U.swap(1,2); U.swap(0,1)
        if a in [4, 11]: U.swap(1,3); U.swap(0,2)
        if a in [7,11,13]:
            for q in range(4): U.x(q)
    return U.to_gate(label=f"{a}^{power} mod 15").control()

# 2. Configuración de Qubits y base elegida
N_COUNT = 4
N_AUX = 4
base_coprima = 7

# 3. Construcción del Circuito Principal
qc = QuantumCircuit(N_COUNT + N_AUX, N_COUNT)

for q in range(N_COUNT):
    qc.h(q)
qc.x(N_COUNT)

for q in range(N_COUNT):
    qc.append(c_amod15(base_coprima, 2**q), [q] + [i+N_COUNT for i in range(N_AUX)])

qc.append(QFTGate(N_COUNT).inverse(), range(N_COUNT))

qc.measure(range(N_COUNT), range(N_COUNT))

simulador = AerSimulator()
circuito_transpilado = transpile(qc, simulador)
shots = 1000

resultados = simulador.run(circuito_transpilado, shots=shots).result().get_counts()

print("\n--- RESULTADOS IDEALES (SIN RUIDO) ---")
print("Distribución de medidas obtenidas:", resultados)

# 0 (0000), 4 (0100), 8 (1000) y 12 (1100).
exitos = (resultados.get('0000', 0) + resultados.get('0100', 0) +
          resultados.get('1000', 0) + resultados.get('1100', 0))